In [1]:
from transformers import AutoTokenizer
from SMOLL2_135M_without_acceleration import LlamaForCausalLM
import torch

# Load model directly
tokenizer =AutoTokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")

dim = 576
num_layers = 30
hidden_dim = 1100
model = LlamaForCausalLM(49152, dim, num_layers, hidden_dim)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Initializing model with vocab_size: 49152


In [19]:
# Load the checkpoint
checkpoint = torch.load("/content/final_checkpoint.pth")

<ipython-input-19-8962325c8482>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/content/final_checkpoint.pth")


In [20]:
checkpoint.keys()

dict_keys(['step', 'model_state_dict', 'optimizer_state_dict', 'loss'])

In [21]:
# Load the state_dict (weights) into the model
model.load_state_dict(checkpoint['model_state_dict'])
#optimizer.load_state_dict(checkpoint['optimizer_state_dict'])


<All keys matched successfully>

In [22]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])  # If you have an optimizer in the checkpoint

In [23]:
# Optionally move model to GPU if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x TransformerBlock(
        (self_attn): SelfAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=True)
          (k_proj): Linear(in_features=576, out_features=576, bias=True)
          (v_proj): Linear(in_features=576, out_features=576, bias=True)
          (o_proj): Linear(in_features=576, out_features=576, bias=True)
          (rotary_emb): RotaryEmbedding()
        )
        (feed_forward): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=576, out_features=1100, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=1100, out_features=576, bias=True)
          )
        )
        (ln1): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_final): LayerNorm((576,), eps=1e-05, eleme

In [9]:
from train import load_input_file_dataset,get_dataloader
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
from torch.amp import autocast  # Correct import

scaler = torch.cuda.amp.GradScaler()

# Modified to assign dataset to the first element of returned value
dataset, tokenizer = load_input_file_dataset(seq_length=750) # assign the first element (input_ids) to dataset

dataloader = get_dataloader(dataset, batch_size=32)

# Assuming `device` is already set to either "cuda" or "cpu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to the correct device
model = model.to(device)

# Ensure optimizer is created after moving the model to the device
optimizer = optim.Adam(model.parameters(), lr=0.001)

step = 0
model.train()
while step < 50:
    for batch in dataloader:

        # Ensure all tensors are on the correct device
        input_ids = batch[0].to(device)
        labels = input_ids[:, 1:].contiguous().to(device)
        inputs = input_ids[:, :-1].contiguous().to(device)

        optimizer.zero_grad()

        criterion = nn.CrossEntropyLoss()

        # Updated autocast usage
        with autocast(device_type=device.type):  # No need for `device_type`
            outputs = model(inputs)
            loss = criterion(outputs.view(-1, 49152), labels.view(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        step += 1
        print(f"The step number is step {step}: {loss.item()}")

<ipython-input-9-6fbe5ec47c77>:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Loading tokenizer...
Tokenizer loaded. Vocab size: 49152
Set pad_token to eos_token
Loading input.txt file...
Tokenizing input text...
Tokenization complete.
The step number is step 1: 4.876454830169678
The step number is step 2: 4.952186107635498
The step number is step 3: 4.878963470458984
The step number is step 4: 4.892857074737549
The step number is step 5: 4.91405725479126
The step number is step 6: 4.9035797119140625
The step number is step 7: 4.882721424102783
The step number is step 8: 4.876523017883301
The step number is step 9: 4.884971618652344
The step number is step 10: 4.891433238983154
The step number is step 11: 4.891271591186523
The step number is step 12: 4.882874965667725
The step number is step 13: 4.877931118011475
The step number is step 14: 4.876288414001465
The step number is step 15: 4.880331516265869
The step number is step 16: 4.883453845977783
The step number is step 17: 4.8825883865356445
The step number is step 18: 4.879138469696045
The step number is ste

In [12]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x TransformerBlock(
        (self_attn): SelfAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=True)
          (k_proj): Linear(in_features=576, out_features=576, bias=True)
          (v_proj): Linear(in_features=576, out_features=576, bias=True)
          (o_proj): Linear(in_features=576, out_features=576, bias=True)
          (rotary_emb): RotaryEmbedding()
        )
        (feed_forward): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=576, out_features=1100, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=1100, out_features=576, bias=True)
          )
        )
        (ln1): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_final): LayerNorm((576,), eps=1e-05, eleme

In [13]:
# prompt: print model parameter count please

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")


Total number of parameters: 134641896


#Inference

In [25]:
# Example input text
input_text = "What is the capital of France?"

your_vocab = tokenizer.get_vocab()

# Create reverse vocab mapping
your_vocab_rev = {idx: token for token, idx in tokenizer.get_vocab().items()}

model.eval()

# Tokenizing the input text
input_ids = torch.tensor([[your_vocab.get(token, 0) for token in input_text.split()]])

# Move input_ids to the same device as the model
input_ids = input_ids.to(device) # This line is added to move input_ids to the device

# Generate a response
output_ids = model.generate(input_ids, max_length=20)

# Decode the output tokens back to text
output_text = " ".join([your_vocab_rev.get(id.item(), '<unk>') for id in output_ids[0]])

print(f"Input: {input_text}")
print(f"Response: {output_text}")

Input: What is the capital of France?
Response: What is the capital of <|endoftext|> Ġguess Ġand Ġsir Ċ su Ċ ĠCitizen Ġsenate Ġrevenge Ġpeople Ġyou Ġin Ċ Ġmatter ? Our Ġhuman ? em ,
